# 26 — Hugging Face export: six tables per member (Milestone 3)

The final submission is one public Hugging Face dataset **per member**, holding the rows of that member's
arXiv ID range (teacher's QA, Milestone 3 question 2). The statement fixes the layout:

- six tables, `papers`, `references`, `elements`, `hierarchy`, `bibtex`, `predictions`, each in
  `data/<table>/` as sharded Parquet;
- flat columns only (strings, integers, one-level lists of strings), no nested objects or JSON strings;
- a dataset card `README.md` whose YAML `configs` has one entry per table, `split: train`,
  `path: data/<table>/*.parquet`; no loading script.

| Section | Functions |
|---|---|
| 0. Setup | loads `20_m3_data` (paths, `MEMBER_RANGES`, `member_of`), `papers_rows` (01), `references_rows` (03) |
| 1. Tables | `HF_SCHEMAS`, `conform`, `ShardWriter`, `export_all` |
| 2. Dataset card | `card_text`, `write_card` |
| 3. Checks | `check_export` |
| 4. Upload | `hf_token`, `default_repo_id`, `upload_member`, `check_hub` |
| 5. Tests | `ExportTests` |
| 6. Run | `EXPORT`, `UPLOAD_MEMBER` |

**Sources** (Drive `MyDrive/arxiv-project/`): `data/raw/<id>/metadata.json` (papers, M1),
`data/raw/<id>/references.json` (references, M1; only references with an arXiv ID become rows),
`data/m2/{elements,hierarchy,bibtex}.parquet` (M2), `data/m3/predictions.parquet` (M3, `23_model`).

**Output** (Drive, **never committed**): `hf/<member>/README.md` and `hf/<member>/data/<table>/train-0000i-of-0000n.parquet`,
plus `logs/hf_export.json` (rows per member and table, rows outside every range).

**How to run** (Colab, CPU): whoever holds the Drive sets `EXPORT = True` and runs all: one pass over each
source builds the four member folders. Then **each member** opens this notebook with their own `HF_TOKEN`
(a *write* token in Colab Secrets), sets `UPLOAD_MEMBER = "TV3"` (their own) and runs all: the folder is
checked, uploaded to `<hf-user>/khdl-arxiv-<range>` and read back with `load_dataset(..., streaming=True)`.
After `23_model` changes `predictions`, `EXPORT_TABLES = ("predictions",)` rebuilds only that table.

## 0. Setup

In [ ]:
# On Colab, when this notebook is opened on its own: clone or update the branch, install the packages and work
# from src/ (the notebooks use relative paths). When another notebook runs it with %run -i, nothing happens here.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("./00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "m3-labeling"   # the Milestone 3 branch
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

if "RUN_TEST" not in globals():
    RUN_TEST = True          # offline tests of section 5 (a toy project in a temporary folder)
EXPORT = False               # True on Colab: build hf/<member>/ from the Drive data (section 6)
EXPORT_MEMBERS = None        # None: the four members; or e.g. ["TV3"]
EXPORT_TABLES = None         # None: the six tables; ("predictions",) after a new 23_model run
UPLOAD_MEMBER = None         # "TV3": upload hf/TV3/ with this Colab's HF_TOKEN, then read it back from the Hub
HF_REPO_ID = None            # None: <your hf user>/khdl-arxiv-<first id>-<last number> (default_repo_id)
MAX_SHARD_MB = 256           # Arrow (uncompressed) size per Parquet shard; zstd files are several times smaller
THREADS = 16                 # parallel reads of metadata.json / references.json (Drive reads are the bottleneck)
globals().update(globals().get("HF_OVERRIDES", {}))

In [ ]:
_RUN_TEST_26 = RUN_TEST
RUN_TEST = False
if "MEMBER_RANGES" not in globals() or "M3_DIR" not in globals():
    get_ipython().run_line_magic("run", "-i ./20_m3_data.ipynb")   # 00_config, 13, MEMBER_RANGES, member_of, M3_DIR
RUN_TEST = _RUN_TEST_26

import re
import shutil
import tempfile
import time
import unittest
from concurrent.futures import ThreadPoolExecutor

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

HF_DIR = PROJECT_ROOT / "hf"                       # hf/<member>/: the folder uploaded to Hugging Face
M2_DIR = DATA_DIR.parent / "m2"                    # data/m2/{elements,hierarchy,bibtex}.parquet (15)
PREDICTIONS_SOURCE = M3_DIR / "predictions.parquet"   # 23_model
HF_EXPORT_LOG = LOG_DIR / "hf_export.json"
CODE_REPO = "https://github.com/thinhelzan842/intro-ds-arxiv-project"
SHARD_RE = re.compile(r"^train-(\d{5})-of-(\d{5})\.parquet$")
print(f"26 loaded | hf: {HF_DIR} | m2: {M2_DIR} | predictions: {PREDICTIONS_SOURCE}")

## 1. Tables

`HF_SCHEMAS` is the explicit schema of each table, column for column as in the statement: `list<string>` for
`authors`, `revised_dates`, `ref_authors`; a dictionary column for `kind`, `entry_type`, `partition`; `int32` for
`version` and the nullable `year`; `nullable=False` where the statement says *non-null*. The three M2 tables
already have this schema (`15_run_parsing`); `conform` casts every batch to it and refuses a null in a
non-null column, so a wrong source fails here and not on the Hub.

`export_all(members, tables)` reads each source **once** and sends every row to the member whose range holds its
`arxiv_id` (`member_of` of `20`): the M2 and predictions Parquet files are streamed in batches, the M1 JSON files
are read with `THREADS` threads. Every `arxiv_id` must be a dotted ID without version (`2605.17338`): any other
form stops the export, since it would silently fall outside every range. The IDs of the source that are in no
member's range are logged (count and examples). `ShardWriter` starts a new shard when
the current one passes `MAX_SHARD_MB` and names them `train-0000i-of-0000n.parquet` at the end (an empty table
still gets one shard with its schema, so its config loads). Each table is written next to the old one and
swapped in at the end, so `EXPORT_TABLES = ("predictions",)` replaces only that table.

In [ ]:
_S, _I32, _LIST = pa.string(), pa.int32(), pa.list_(pa.string())
_DICT = pa.dictionary(pa.int32(), pa.string())
HF_SCHEMAS = {
    "papers": pa.schema([
        pa.field("arxiv_id", _S, nullable=False), pa.field("title", _S), pa.field("authors", _LIST),
        pa.field("submission_date", _S), pa.field("revised_dates", _LIST), pa.field("venue", _S)]),
    "references": pa.schema([
        pa.field("arxiv_id", _S, nullable=False), pa.field("ref_arxiv_id", _S, nullable=False),
        pa.field("ref_title", _S), pa.field("ref_authors", _LIST), pa.field("ref_submission_date", _S),
        pa.field("ref_s2_id", _S)]),
    "elements": pa.schema([
        pa.field("element_id", _S, nullable=False), pa.field("arxiv_id", _S, nullable=False),
        pa.field("kind", _DICT), pa.field("content", _S)]),
    "hierarchy": pa.schema([
        pa.field("arxiv_id", _S, nullable=False), pa.field("version", _I32),
        pa.field("element_id", _S, nullable=False), pa.field("parent_id", _S, nullable=False)]),
    "bibtex": pa.schema([
        pa.field("arxiv_id", _S, nullable=False), pa.field("citation_key", _S, nullable=False),
        pa.field("entry_type", _DICT), pa.field("title", _S), pa.field("authors", _LIST), pa.field("year", _I32),
        pa.field("doi", _S), pa.field("journal", _S), pa.field("url", _S)]),
    "predictions": pa.schema(
        [pa.field("arxiv_id", _S, nullable=False), pa.field("partition", _DICT),
         pa.field("citation_key", _S, nullable=False), pa.field("groundtruth_arxiv_id", _S)]
        + [pa.field(f"cand_{i}", _S) for i in range(1, 6)]),
}
HF_TABLES = tuple(HF_SCHEMAS)
M1_TABLES = ("papers", "references")
M2_TABLES = ("elements", "hierarchy", "bibtex")
PARTITIONS = ("train", "valid", "test")
_FLAT_TYPES = (_S, _I32, _LIST, _DICT)
assert all(f.type in _FLAT_TYPES for s in HF_SCHEMAS.values() for f in s), "a column is not flat"
assert [tuple(HF_SCHEMAS[t].names) for t in M2_TABLES] == [ELEMENT_COLUMNS, HIERARCHY_COLUMNS, BIBTEX_COLUMNS]


def conform(table, schema, where=""):
    """`table` with exactly the columns and types of `schema`; ValueError on a missing column or a null in a
    non-null column."""
    missing = [n for n in schema.names if n not in table.column_names]
    if missing:
        raise ValueError(f"{where}: columns {missing} missing (has {table.column_names})")
    arrays = []
    for field in schema:
        column = table.column(field.name)
        if pa.types.is_dictionary(field.type) and not pa.types.is_dictionary(column.type):
            column = pc.cast(column, field.type.value_type).dictionary_encode()
        column = column.cast(field.type)
        if not field.nullable and column.null_count:
            raise ValueError(f"{where}: {column.null_count} nulls in the non-null column {field.name}")
        arrays.append(column)
    return pa.Table.from_arrays(arrays, schema=schema)


class ShardWriter:
    """Parquet shards of one table in `folder`, named train-0000i-of-0000n.parquet by close()."""

    def __init__(self, folder, schema, max_bytes):
        self.folder, self.schema, self.max_bytes = Path(folder), schema, max_bytes
        shutil.rmtree(self.folder, ignore_errors=True)      # left by an interrupted export
        self.folder.mkdir(parents=True)
        self.parts, self.writer, self.size, self.rows = [], None, 0, 0

    def _next(self):
        if self.writer is not None:
            self.writer.close()
        path = self.folder / f"part-{len(self.parts):05d}.parquet"
        self.parts.append(path)
        self.writer = pq.ParquetWriter(path, self.schema, compression="zstd")
        self.size = 0

    def write(self, table):
        """Append `table`, cut into slices so that no shard grows much past max_bytes."""
        if table.num_rows == 0:
            return
        per_row = max(1, table.nbytes // table.num_rows)
        start = 0
        while start < table.num_rows:
            if self.writer is None or self.size >= self.max_bytes:
                self._next()
            room = max(1, (self.max_bytes - self.size) // per_row)
            piece = table.slice(start, room)
            self.writer.write_table(piece, row_group_size=100_000)
            self.size += piece.num_rows * per_row
            self.rows += piece.num_rows
            start += piece.num_rows

    def close(self):
        if self.writer is None:              # an empty table still gets one shard with its schema
            self._next()
            self.writer.write_table(self.schema.empty_table())
        self.writer.close()
        n = len(self.parts)
        names = []
        for i, path in enumerate(self.parts):
            name = f"train-{i:05d}-of-{n:05d}.parquet"
            path.rename(self.folder / name)
            names.append(name)
        return names


def paper_ids(data_dir=None):
    """Dotted IDs of the paper folders of data/raw (2605-17338 -> 2605.17338), in ID order."""
    data_dir = Path(data_dir or DATA_DIR)
    names = [n for n in os.listdir(data_dir) if re.fullmatch(r"\d{4}-\d{4,5}", n)]
    return sorted(n.replace("-", ".") for n in names)


def _m1_functions(data_dir):
    """papers_rows (01) and references_rows (03) reading `data_dir` (their own namespace)."""
    space = {**globals(), "DATA_DIR": Path(data_dir)}
    load_functions("01_discovery", ("papers_rows",), space)
    load_functions("03_references", ("references_path", "_saved_references", "references_rows"), space)
    return space["papers_rows"], space["references_rows"]


def _m1_rows(table, ids, data_dir, threads):
    papers_rows, references_rows = _m1_functions(data_dir)
    read = (lambda chunk: papers_rows(chunk, data_dir)) if table == "papers" else references_rows
    chunks = [ids[i:i + 250] for i in range(0, len(ids), 250)]

    def read_chunk(chunk):
        try:
            return read(chunk)
        except ValueError:   # a JSON file torn by a Drive disconnect: name it
            bad = []
            for arxiv_id in chunk:
                try:
                    read([arxiv_id])
                except ValueError as e:
                    bad.append(f"{to_folder_id(arxiv_id)}: {e}")
            if not bad:      # not a torn file: keep the original error
                raise
            raise ValueError(f"unreadable {table} JSON files (run the crawl of these papers again): {bad[:5]}") from None

    with ThreadPoolExecutor(max(1, threads)) as pool:
        return [row for rows in pool.map(read_chunk, chunks) for row in rows]


def canonical_id(arxiv_id):
    """True for a dotted arXiv ID without version ('2605.17338'), the form of every table."""
    return isinstance(arxiv_id, str) and bool(NEW_ID_RE.match(arxiv_id))


def _split_by_member(table, members, where=""):
    """{member: rows of `table` whose arxiv_id is in its range}, and the IDs in no member's range.
    ValueError on an arxiv_id that is not a dotted ID without version."""
    ids = table.column("arxiv_id")
    unique = pc.unique(ids).to_pylist()
    odd = [a for a in unique if not canonical_id(a)]
    if odd:
        raise ValueError(f"{where}: {len(odd)} arxiv_id values not of the form 2605.17338, e.g. {odd[:5]}")
    owner = {a: member_of(a) for a in unique}
    out = {}
    for member in members:
        mine = pa.array([a for a, m in owner.items() if m == member], pa.string())
        out[member] = table.filter(pc.is_in(ids, value_set=mine))
    return out, {a for a, m in owner.items() if m is None}


def _source_batches(table, data_dir, m2_dir, predictions_path, threads):
    """Batches (pyarrow Tables) of one source table, and its total number of rows."""
    if table in M1_TABLES:
        rows = _m1_rows(table, paper_ids(data_dir), data_dir, threads)
        whole = pa.Table.from_pylist(rows, schema=HF_SCHEMAS[table]) if rows else HF_SCHEMAS[table].empty_table()
        return [whole], whole.num_rows
    path = Path(m2_dir) / f"{table}.parquet" if table in M2_TABLES else Path(predictions_path)
    if not path.is_file():
        raise FileNotFoundError(f"{path} is missing: {'run 15' if table in M2_TABLES else 'run 23_model'} first, "
                                f"or leave {table!r} out of EXPORT_TABLES")
    source = pq.ParquetFile(path)
    names = [n for n in HF_SCHEMAS[table].names if n in source.schema_arrow.names]
    batches = (pa.Table.from_batches([b]) for b in source.iter_batches(batch_size=200_000, columns=names))
    return batches, source.metadata.num_rows


def _swap(new, old):
    """Replace the folder `old` by `new` (the old export of a table is kept until the new one is complete)."""
    if old.exists():
        shutil.rmtree(old)
    new.rename(old)


def export_all(members=None, tables=None, out_root=None, data_dir=None, m2_dir=None, predictions_path=None,
               max_shard_bytes=None, threads=None, log_path=None):
    """Write out_root/<member>/data/<table>/ for every member and table, then each member's dataset card."""
    t0 = time.time()
    members = list(members or MEMBER_RANGES)
    tables = tuple(tables or HF_TABLES)
    unknown = [t for t in tables if t not in HF_SCHEMAS] + [m for m in members if m not in MEMBER_RANGES]
    if unknown:
        raise ValueError(f"unknown tables or members: {unknown}")
    out_root = Path(out_root or HF_DIR)
    data_dir, m2_dir = Path(data_dir or DATA_DIR), Path(m2_dir or M2_DIR)
    predictions_path = Path(predictions_path or PREDICTIONS_SOURCE)
    max_bytes = max_shard_bytes or MAX_SHARD_MB * 1024 * 1024
    threads = THREADS if threads is None else threads
    report = {"members": members, "tables": {}}
    for table in tables:
        t1 = time.time()
        schema = HF_SCHEMAS[table]
        writers = {m: ShardWriter(out_root / m / "data" / f"{table}.partial", schema, max_bytes) for m in members}
        batches, source_rows = _source_batches(table, data_dir, m2_dir, predictions_path, threads)
        outside, n_read = set(), 0
        for batch in batches:
            batch = conform(batch, schema, where=f"{table} source")
            n_read += batch.num_rows
            parts, out = _split_by_member(batch, members, where=f"{table} source")
            outside |= out
            for member, part in parts.items():
                writers[member].write(part)
        if n_read != source_rows:
            raise ValueError(f"{table}: read {n_read:,} rows, the source has {source_rows:,}")
        rows = {}
        for member, writer in writers.items():
            shards = writer.close()
            _swap(writer.folder, out_root / member / "data" / table)
            rows[member] = {"rows": writer.rows, "shards": len(shards)}
        not_exported = source_rows - sum(v["rows"] for v in rows.values())   # other members' rows + outside
        report["tables"][table] = {
            "source_rows": source_rows, "members": rows, "rows_not_exported": not_exported,
            "papers_outside_ranges": len(outside), "outside_examples": sorted(outside)[:10],
            "seconds": round(time.time() - t1, 1)}
        print(f"[26] {table}: {source_rows:,} source rows | " + ", ".join(
            f"{m} {v['rows']:,} ({v['shards']} shards)" for m, v in rows.items())
            + f" | not exported {not_exported:,} | papers outside every range {len(outside)} {sorted(outside)[:3]}")
    for member in members:
        write_card(out_root / member, member)
    report["seconds"] = round(time.time() - t0, 1)
    log_path = Path(log_path or HF_EXPORT_LOG)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old = read_json(log_path, {}) or {}
    old.setdefault("runs", []).append({**report, "at": time.strftime("%Y-%m-%d %H:%M:%S")})
    write_json(log_path, old)
    return report

## 2. Dataset card

`write_card` writes `README.md` from what is in the folder: the YAML front matter lists one config per table
folder (`split: train`, `path: data/<table>/*.parquet`), and the body gives the ID range, the row and shard
counts read from the Parquet footers, every column with its type, and how to load a table. The card has no
personal names: the member is named by its role and ID range.

In [ ]:
COLUMN_DOCS = {
    "papers": {"arxiv_id": "arXiv ID without version", "title": "title (arXiv API)",
               "authors": "author names", "submission_date": "v1 submission date (ISO 8601)",
               "revised_dates": "dates of v2..vN", "venue": "journal reference, if any"},
    "references": {"arxiv_id": "citing paper", "ref_arxiv_id": "arXiv ID of the reference (old-style IDs kept)",
                   "ref_title": "title (Semantic Scholar)", "ref_authors": "author names",
                   "ref_submission_date": "publication date of the reference", "ref_s2_id": "Semantic Scholar ID"},
    "elements": {"element_id": "unique element ID", "arxiv_id": "paper", "kind": "document, section, sentence, "
                 "formula, figure, ...", "content": "cleaned text (identical content across versions is one row)"},
    "hierarchy": {"arxiv_id": "paper", "version": "arXiv version", "element_id": "child element",
                  "parent_id": "parent element"},
    "bibtex": {"arxiv_id": "paper", "citation_key": "entry key after deduplication", "entry_type": "BibTeX type",
               "title": "title", "authors": "author names", "year": "year", "doi": "DOI", "journal": "journal",
               "url": "URL"},
    "predictions": {"arxiv_id": "paper", "partition": "train, valid or test", "citation_key": "BibTeX entry",
                    "groundtruth_arxiv_id": "true reference (an arXiv reference of the paper)",
                    **{f"cand_{i}": f"rank {i} candidate reference" for i in range(1, 6)}},
}
TABLE_DOCS = {
    "papers": "arXiv metadata of every paper of the range (Milestone 1).",
    "references": "references of each paper from Semantic Scholar; only those with an arXiv ID (Milestone 1).",
    "elements": "nodes of the LaTeX hierarchy, shared across versions (Milestone 2).",
    "hierarchy": "child -> parent edges of each version (Milestone 2).",
    "bibtex": "BibTeX entries after deduplication and merging (Milestone 2).",
    "predictions": "top-5 candidate references of every BibTeX entry of the train, valid and test papers (Milestone 3).",
}
assert {t: tuple(d) for t, d in COLUMN_DOCS.items()} == {t: tuple(s.names) for t, s in HF_SCHEMAS.items()}


def _type_name(t):
    if pa.types.is_dictionary(t):
        return "string (dictionary)"
    return {"list<item: string>": "list<string>"}.get(str(t), str(t))


def _table_stats(folder):
    files = sorted(Path(folder).glob("*.parquet"))
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files), len(files)


def card_text(member_dir, member, repo_id=None):
    member_dir = Path(member_dir)
    first, last = MEMBER_RANGES[member]
    present = [t for t in HF_TABLES if (member_dir / "data" / t).is_dir()]
    yaml = ["---", f"pretty_name: arXiv papers {first} to {last}", "language:", "- en", "tags:", "- arxiv",
            "- latex", "- citation-matching", "configs:"]
    for t in present:
        yaml += [f"- config_name: {t}", "  data_files:", "  - split: train", f"    path: data/{t}/*.parquet"]
    yaml.append("---")
    name = repo_id or "<this repo>"
    body = [
        f"# arXiv papers {first} to {last}", "",
        f"Data of the arXiv IDs {first} to {last}, crawled and processed by group 10 of the course "
        f"Introduction to Data Science (HCMUS), member {member[-1]}. Code: {CODE_REPO}", "",
        "| Table | Rows | Shards | Content |", "|---|---|---|---|"]
    for t in present:
        rows, shards = _table_stats(member_dir / "data" / t)
        body.append(f"| `{t}` | {rows:,} | {shards} | {TABLE_DOCS[t]} |")
    body += ["", "All IDs are arXiv IDs without version (`2605.17338`); a reference may have an old-style ID "
             "(`hep-th/9901001`).", ""]
    for t in present:
        body += [f"## `{t}`", "", "| Column | Type | Content |", "|---|---|---|"]
        body += [f"| `{f.name}` | {_type_name(f.type)}{'' if f.nullable else ', non-null'} | {COLUMN_DOCS[t][f.name]} |"
                 for f in HF_SCHEMAS[t]]
        body.append("")
    body += ["## Loading", "", "```python", "from datasets import load_dataset",
             f'papers = load_dataset("{name}", "papers", split="train")', "```", ""]
    if "predictions" in present:
        body += ["In `predictions`, every row is an entry with a true reference (`groundtruth_arxiv_id`): an "
                 "entry without one is not a sample; `cand_k` is null when the paper has fewer than k references. "
                 "MRR@5 is computed on the `test` rows.", ""]
    return "\n".join(yaml + body)


def write_card(member_dir, member, repo_id=None):
    path = Path(member_dir) / "README.md"
    path.write_text(card_text(member_dir, member, repo_id), encoding="utf-8")
    return path

## 3. Checks

`check_export(member_dir, member)` reads the folder back the way the Hub will and lists every problem:
the card configs match the table folders; shard names are `train-0000i-of-0000n` with no gap; each shard has
exactly the schema of `HF_SCHEMAS` (names, types, non-null flags, so no null in a non-null column); every
`arxiv_id` is a dotted ID without version in the member's range. Between tables: every paper of a table is in
`papers`; `predictions` has only `train`/`valid`/`test`, one row per (`arxiv_id`, `citation_key`), every such
pair is in `bibtex`, and every ground truth and candidate is a reference of the same paper in `references`
(so a `predictions` re-export from another model run cannot be uploaded against stale tables). `upload_member` runs it first and refuses to upload a folder that fails it.

In [ ]:
def _front_matter(text):
    m = re.match(r"^---\n(.*?)\n---\n", text, re.S)
    if not m:
        return None
    configs, current = {}, None
    for line in m.group(1).splitlines():
        if line.startswith("- config_name: "):
            current = line.split(": ", 1)[1].strip()
        elif line.strip().startswith("path: ") and current:
            configs[current] = line.split(": ", 1)[1].strip()
    return configs


def _cross_checks(seen, problems):
    """Checks between tables (only those read): papers known, predictions keys in bibtex, candidates in references."""
    def pairs(table, a, b):
        return set(zip(table.column(a).to_pylist(), table.column(b).to_pylist()))

    papers = seen.get("papers")
    if papers is not None:
        known = set(papers.column("arxiv_id").to_pylist())
        for t, table in seen.items():
            if t != "papers" and table is not None:
                unknown = set(pc.unique(table.column("arxiv_id")).to_pylist()) - known
                if unknown:
                    problems.append(f"{t}: {len(unknown)} papers not in papers, e.g. {sorted(unknown)[:3]}")
    pred = seen.get("predictions")
    if pred is None:
        return
    keys = list(zip(pred.column("arxiv_id").to_pylist(), pred.column("citation_key").to_pylist()))
    if len(keys) != len(set(keys)):
        problems.append(f"predictions: {len(keys) - len(set(keys))} duplicate (arxiv_id, citation_key)")
    bad = set(pc.unique(pc.cast(pred.column("partition"), pa.string())).to_pylist()) - set(PARTITIONS)
    if bad:
        problems.append(f"predictions: partitions {sorted(map(str, bad))}")
    if seen.get("bibtex") is not None:
        missing = set(keys) - pairs(seen["bibtex"], "arxiv_id", "citation_key")
        if missing:
            problems.append(f"predictions: {len(missing)} entries not in bibtex, e.g. {sorted(missing)[:3]}")
    if seen.get("references") is not None:
        refs = pairs(seen["references"], "arxiv_id", "ref_arxiv_id")
        foreign = {(a, r) for c in ["groundtruth_arxiv_id"] + [f"cand_{i}" for i in range(1, 6)]
                   for a, r in zip(pred.column("arxiv_id").to_pylist(), pred.column(c).to_pylist())
                   if r is not None and (a, r) not in refs}
        if foreign:
            problems.append(f"predictions: {len(foreign)} candidates or ground truths not in the paper's "
                            f"references, e.g. {sorted(foreign)[:3]}")


def check_export(member_dir, member, tables=None):
    """{table: rows} of a member folder; ValueError listing every problem."""
    member_dir = Path(member_dir)
    tables = tuple(tables or HF_TABLES)
    problems, rows, seen = [], {}, {}
    card = member_dir / "README.md"
    configs = _front_matter(card.read_text(encoding="utf-8")) if card.is_file() else None
    if configs is None:
        problems.append("README.md with a YAML front matter is missing")
        configs = {}
    for t in tables:
        folder = member_dir / "data" / t
        if not folder.is_dir():
            problems.append(f"{t}: data/{t}/ is missing")
            continue
        if configs.get(t) != f"data/{t}/*.parquet":
            problems.append(f"{t}: card config path {configs.get(t)!r}")
        files = sorted(p.name for p in folder.iterdir())
        found = [SHARD_RE.match(n) for n in files]
        if not files or not all(found) or {int(m.group(2)) for m in found} != {len(files)} \
                or sorted(int(m.group(1)) for m in found) != list(range(len(files))):
            problems.append(f"{t}: shard names {files[:4]}")
        schema, n = HF_SCHEMAS[t], 0
        columns = {"bibtex": ["arxiv_id", "citation_key"], "references": ["arxiv_id", "ref_arxiv_id"],
                   "predictions": schema.names}.get(t, ["arxiv_id"])
        parts = []
        for name in files:
            path = folder / name
            got = pq.read_schema(path).remove_metadata()
            if not got.equals(schema):   # names, types and non-null flags (a non-null column cannot hold a null)
                problems.append(f"{t}/{name}: schema {got} differs from {schema}")
                continue
            data = pq.read_table(path, columns=columns)
            n += data.num_rows
            parts.append(data)
            ids = pc.unique(data.column("arxiv_id")).to_pylist()
            foreign = {a for a in ids if not canonical_id(a) or member_of(a) != member}
            if foreign:
                problems.append(f"{t}/{name}: {len(foreign)} arxiv_id outside {member} or not of the form "
                                f"2605.17338, e.g. {sorted(foreign)[:3]}")
        rows[t] = n
        seen[t] = pa.concat_tables(parts) if parts else None
    _cross_checks(seen, problems)
    if problems:
        raise ValueError(f"hf/{member}:\n- " + "\n- ".join(problems))
    return rows

## 4. Upload

Each member uploads **their own** folder with their own token (teacher's QA, Milestone 3 question 2):
`HF_TOKEN`, a *write* token from huggingface.co/settings/tokens, goes in Colab Secrets (key icon, notebook access
on). `upload_member` checks the folder, creates the public dataset repo if needed, uploads the folder in one
commit (old shards under `data/` that are not in the new export are deleted), and `check_hub` reads one row of
every config with `load_dataset(..., streaming=True)`. The link to paste on Moodle is printed.

In [ ]:
def hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token and ON_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception as e:   # no secret, or notebook access off
            print(f"[26] Colab secret HF_TOKEN not read: {type(e).__name__}: {e}")
    if not token:
        raise RuntimeError("HF_TOKEN missing: add a write token to Colab Secrets (key icon), notebook access on")
    return token


def default_repo_id(user, member):
    """<user>/khdl-arxiv-2605.17338-22337 for TV3."""
    first, last = MEMBER_RANGES[member]
    return f"{user}/khdl-arxiv-{first}-{last.split('.')[1]}"


def upload_member(member, repo_id=None, token=None, out_root=None):
    """Check hf/<member>/, upload it to a public dataset repo; returns (repo_id, url)."""
    member_dir = Path(out_root or HF_DIR) / member
    check_export(member_dir, member)                      # the six tables, or nothing is uploaded
    from huggingface_hub import HfApi
    token = token or hf_token()
    api = HfApi(token=token)
    repo_id = repo_id or default_repo_id(api.whoami()["name"], member)
    write_card(member_dir, member, repo_id)                # the loading example names the real repo
    api.create_repo(repo_id, repo_type="dataset", private=False, exist_ok=True)
    try:   # a repo made private earlier stays private with exist_ok: the statement asks for a public dataset
        api.update_repo_settings(repo_id, repo_type="dataset", private=False)
    except AttributeError:   # huggingface_hub < 0.25
        api.update_repo_visibility(repo_id, private=False, repo_type="dataset")
    api.upload_folder(repo_id=repo_id, repo_type="dataset", folder_path=str(member_dir),
                      commit_message=f"{member}: six tables of {' to '.join(MEMBER_RANGES[member])}",
                      ignore_patterns=["*.partial/*"], delete_patterns=["data/**"])
    url = f"https://huggingface.co/datasets/{repo_id}"
    print(f"[26] uploaded {member_dir} -> {url}")
    return repo_id, url


def check_hub(repo_id, tables=None, token=None):
    """{table: 'ok' | problem}: one streamed row of every config, its columns compared with HF_SCHEMAS."""
    from datasets import load_dataset
    out = {}
    for t in tables or HF_TABLES:
        try:
            row = next(iter(load_dataset(repo_id, t, split="train", streaming=True, token=token)))
            out[t] = "ok" if list(row) == HF_SCHEMAS[t].names else f"columns {list(row)}"
        except StopIteration:
            out[t] = "empty table"
        except Exception as e:
            out[t] = f"{type(e).__name__}: {e}"
    print(f"[26] check_hub {repo_id}: {out}")
    return out

## 5. Tests

A toy project in a temporary folder: papers of TV3, TV4 and one paper outside every range, with
`metadata.json`, `references.json`, the three M2 tables and a predictions table. The tests check that each
member gets only its papers (and the paper outside every range is logged), the schema and the card,
sharding, an empty table, that a versioned ID stops the export, that `check_export` catches a foreign paper, a
nullable key column, a wrong column, a predictions key missing from bibtex and a candidate that is not a reference, that
re-exporting `predictions` alone keeps the other tables, and that `upload_member` refuses an incomplete folder
before any network call.

In [ ]:
def _toy_project(root):
    """data/raw, data/m2, data/m3 of 4 papers: 2 of TV3, 1 of TV4, 1 outside every range."""
    root = Path(root)
    papers = {"2605.17400": 2, "2605.17401": 1, "2605.08000": 1, "2310.06825": 1}
    raw = root / "data" / "raw"
    for arxiv_id, versions in papers.items():
        folder = raw / to_folder_id(arxiv_id)
        folder.mkdir(parents=True)
        write_json(folder / "metadata.json", {
            "id": arxiv_id, "status": "ok", "versions": versions, "title": f"Paper {arxiv_id}",
            "authors": ["A. One", "B. Two"], "submitted": "2026-05-01T00:00:00+00:00",
            "revised": ["2026-05-09T00:00:00+00:00"] * (versions - 1), "venue": None})
        write_json(folder / "references.json", {"id": arxiv_id, "status": "ok", "references": [
            {"arxiv_id": "2301.00001", "title": "R1", "authors": ["C. Three"], "submission_date": "2023-01-01",
             "s2_id": "s1"},
            {"arxiv_id": "hep-th/9901001", "title": "R2", "authors": [], "submission_date": None, "s2_id": "s2"},
            {"arxiv_id": None, "title": "no arXiv id", "authors": [], "submission_date": None, "s2_id": "s3"}]})
    m2 = root / "data" / "m2"
    m2.mkdir(parents=True)
    elements, hierarchy, bibtex = [], [], []
    for arxiv_id in papers:
        doc = f"{to_folder_id(arxiv_id)}#doc"
        elements += [{"element_id": doc, "arxiv_id": arxiv_id, "kind": "document", "content": ""},
                     {"element_id": doc + "-s1", "arxiv_id": arxiv_id, "kind": "sentence", "content": "Hi."}]
        hierarchy.append({"arxiv_id": arxiv_id, "version": 1, "element_id": doc + "-s1", "parent_id": doc})
        bibtex += [{"arxiv_id": arxiv_id, "citation_key": k, "entry_type": "article", "title": "T", "authors": ["X"],
                    "year": 2020 if k == "a" else None, "doi": None, "journal": None, "url": None} for k in ("a", "b")]
    for name, rows in (("elements", elements), ("hierarchy", hierarchy), ("bibtex", bibtex)):
        pq.write_table(pa.Table.from_pylist(rows, schema=HF_SCHEMAS[name]), m2 / f"{name}.parquet")
    m3 = root / "data" / "m3"
    m3.mkdir(parents=True)
    pred = [{"arxiv_id": a, "partition": p, "citation_key": k, "groundtruth_arxiv_id": "2301.00001" if k == "a" else None,
             "cand_1": "2301.00001", "cand_2": "hep-th/9901001", "cand_3": None, "cand_4": None, "cand_5": None}
            for a, p in (("2605.17400", "test"), ("2605.17401", "train"), ("2605.08000", "valid")) for k in ("a", "b")]
    pq.write_table(pa.Table.from_pylist(pred, schema=pa.schema([pa.field(f.name, pa.string())
                                                                for f in HF_SCHEMAS["predictions"]])),
                   m3 / "predictions.parquet")   # plain strings, as 23_model writes them
    return root


class ExportTests(unittest.TestCase):
    def setUp(self):
        self.tmp = tempfile.mkdtemp(prefix="hf-test-")
        self.root = _toy_project(self.tmp)
        self.out = self.root / "hf"

    def tearDown(self):
        shutil.rmtree(self.tmp, ignore_errors=True)

    def export(self, **kw):
        args = dict(out_root=self.out, data_dir=self.root / "data" / "raw", m2_dir=self.root / "data" / "m2",
                    predictions_path=self.root / "data" / "m3" / "predictions.parquet", threads=2,
                    log_path=self.root / "logs" / "hf_export.json")
        return export_all(**{**args, **kw})

    def test_each_member_gets_only_its_papers(self):
        report = self.export()
        rows = check_export(self.out / "TV3", "TV3")
        self.assertEqual(rows["papers"], 2)
        self.assertEqual(rows["references"], 4)               # the reference without an arXiv ID is not a row
        self.assertEqual(rows["predictions"], 4)
        self.assertEqual(check_export(self.out / "TV4", "TV4")["bibtex"], 2)
        self.assertEqual(check_export(self.out / "TV1", "TV1")["elements"], 0)
        self.assertEqual(report["tables"]["papers"]["outside_examples"], ["2310.06825"])
        self.assertEqual(report["tables"]["elements"]["rows_not_exported"], 2)   # the paper outside every range
        refs = pq.read_table(self.out / "TV3" / "data" / "references").column("ref_arxiv_id").to_pylist()
        self.assertIn("hep-th/9901001", refs)

    def test_schema_types_and_card(self):
        self.export(members=["TV3"])
        for t in HF_TABLES:
            files = sorted((self.out / "TV3" / "data" / t).iterdir())
            self.assertEqual([f.name for f in files], ["train-00000-of-00001.parquet"])
            self.assertTrue(pq.read_schema(files[0]).remove_metadata().equals(HF_SCHEMAS[t]), t)
        part = pq.read_table(self.out / "TV3" / "data" / "predictions").column("partition")
        self.assertTrue(pa.types.is_dictionary(part.type))
        text = (self.out / "TV3" / "README.md").read_text(encoding="utf-8")
        self.assertEqual(_front_matter(text), {t: f"data/{t}/*.parquet" for t in HF_TABLES})
        try:
            import yaml
        except ImportError:
            yaml = None
        if yaml:
            meta = yaml.safe_load(text.split("---")[1])
            self.assertEqual([c["config_name"] for c in meta["configs"]], list(HF_TABLES))
            self.assertEqual(meta["configs"][0]["data_files"], [{"split": "train", "path": "data/papers/*.parquet"}])
        self.assertIn("2605.17338 to 2605.22337", text)

    def test_sharding_keeps_every_row(self):
        self.export(members=["TV3"], max_shard_bytes=1)
        files = sorted(p.name for p in (self.out / "TV3" / "data" / "elements").iterdir())
        self.assertGreater(len(files), 1)
        self.assertEqual(files[-1], f"train-{len(files) - 1:05d}-of-{len(files):05d}.parquet")
        self.assertEqual(check_export(self.out / "TV3", "TV3")["elements"], 4)

    def test_check_export_catches_problems(self):
        self.export(members=["TV3"])
        folder = self.out / "TV3" / "data"
        bad = pa.Table.from_pylist([{"arxiv_id": "2605.08000", "citation_key": "z", "entry_type": "misc"}],
                                   schema=HF_SCHEMAS["bibtex"])
        pq.write_table(bad, folder / "bibtex" / "train-00000-of-00001.parquet")
        nulls = pa.Table.from_arrays([pa.array(["2605.17400"]), pa.array([1], pa.int32()), pa.array(["e"]),
                                      pa.array([None], pa.string())], names=list(HF_SCHEMAS["hierarchy"].names))
        pq.write_table(nulls, folder / "hierarchy" / "train-00000-of-00001.parquet")
        pq.write_table(pa.table({"arxiv_id": ["2605.17400"]}), folder / "papers" / "train-00000-of-00001.parquet")
        with self.assertRaises(ValueError) as caught:
            check_export(self.out / "TV3", "TV3")
        text = str(caught.exception)
        self.assertIn("outside TV3 or not of the form", text)
        self.assertIn("papers/train-00000-of-00001.parquet: schema", text)
        self.assertIn("hierarchy/train-00000-of-00001.parquet: schema", text)   # nullable parent_id

    def test_versioned_or_folder_ids_stop_the_export(self):
        m2 = self.root / "data" / "m2" / "bibtex.parquet"
        table = pq.read_table(m2)
        ids = table.column("arxiv_id").to_pylist()
        ids[0] = "2605.17400v2"
        pq.write_table(table.set_column(0, "arxiv_id", pa.array(ids, pa.string())).cast(HF_SCHEMAS["bibtex"]), m2)
        with self.assertRaises(ValueError) as caught:
            self.export(members=["TV3"], tables=["bibtex"])
        self.assertIn("2605.17400v2", str(caught.exception))

    def test_cross_table_checks(self):
        self.export(members=["TV3"])
        folder = self.out / "TV3" / "data" / "predictions"
        pred = pq.read_table(folder / "train-00000-of-00001.parquet")
        keys = pred.column("citation_key").to_pylist()
        keys[0] = "renamed"                                   # a key that bibtex does not have
        cands = pred.column("cand_1").to_pylist()
        cands[1] = "2399.99999"                               # not a reference of the paper
        pred = pred.set_column(2, pa.field("citation_key", pa.string(), nullable=False), pa.array(keys, pa.string()))
        pred = pred.set_column(4, pa.field("cand_1", pa.string()), pa.array(cands, pa.string()))
        pq.write_table(pred, folder / "train-00000-of-00001.parquet")
        with self.assertRaises(ValueError) as caught:
            check_export(self.out / "TV3", "TV3")
        text = str(caught.exception)
        self.assertIn("entries not in bibtex", text)
        self.assertIn("not in the paper's references", text)

    def test_conform_casts_and_refuses_nulls(self):
        t = conform(pa.table({"arxiv_id": ["a"], "partition": ["test"], "citation_key": ["k"],
                              "groundtruth_arxiv_id": [None], **{f"cand_{i}": [None] for i in range(1, 6)}}),
                    HF_SCHEMAS["predictions"])
        self.assertTrue(pa.types.is_dictionary(t.schema.field("partition").type))
        with self.assertRaises(ValueError):
            conform(pa.table({"arxiv_id": ["a"], "partition": ["test"], "citation_key": [None],
                              "groundtruth_arxiv_id": [None], **{f"cand_{i}": [None] for i in range(1, 6)}}),
                    HF_SCHEMAS["predictions"])

    def test_predictions_alone_keeps_the_other_tables(self):
        self.export(members=["TV3"], tables=[t for t in HF_TABLES if t != "predictions"])
        with self.assertRaises(ValueError):
            check_export(self.out / "TV3", "TV3")             # five tables: not ready to upload
        with self.assertRaises(ValueError):
            upload_member("TV3", repo_id="x/y", token="not-used", out_root=self.out)   # refused before the network
        self.export(members=["TV3"], tables=["predictions"])
        self.assertEqual(check_export(self.out / "TV3", "TV3")["elements"], 4)

    def test_missing_source_says_what_to_run(self):
        with self.assertRaises(FileNotFoundError) as caught:
            self.export(members=["TV3"], tables=["predictions"], predictions_path=self.root / "none.parquet")
        self.assertIn("23_model", str(caught.exception))

    def test_default_repo_id(self):
        self.assertEqual(default_repo_id("someone", "TV3"), "someone/khdl-arxiv-2605.17338-22337")


if RUN_TEST:
    _result = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(ExportTests))
    assert _result.wasSuccessful(), "26 tests failed"

## 6. Run

1. **Export** (whoever holds the Drive): `EXPORT = True`, run all. Prints the rows of every table per member and
   the rows outside the four ranges, writes `hf/<member>/` and `logs/hf_export.json`, and checks each folder.
2. **Upload** (each member, own Colab, own `HF_TOKEN`): `UPLOAD_MEMBER = "TV3"` (your own), run all. Paste the
   printed link on Moodle.

In [ ]:
if EXPORT:
    HF_REPORT = export_all(EXPORT_MEMBERS, EXPORT_TABLES)
    for _member in HF_REPORT["members"]:
        try:
            print(f"[26] check {_member}: {check_export(HF_DIR / _member, _member)}")
        except ValueError as e:   # e.g. predictions not exported yet
            print(f"[26] {e}")
if UPLOAD_MEMBER:
    HF_REPO_ID, HF_URL = upload_member(UPLOAD_MEMBER, HF_REPO_ID)
    HF_HUB_CHECK = check_hub(HF_REPO_ID)
    print(f"Moodle link for {UPLOAD_MEMBER}: {HF_URL}")